# optimizer-state-tensor-buffers — faded example 2: Allocate Adam m and v buffers without aliasing

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-state-tensor-buffers`. The last cell reports your progress on the `Optimizer: Per-param state buffers` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Per-param state buffers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-state-tensor-buffers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-state-tensor-buffers"
DD_SUBTOPIC = "Optimizer: Per-param state buffers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Adam requires two separate buffer lists: `m` for first moments and `v` for second moments. A common mistake is `self.v = self.m` — this creates an alias so both names point to the same list of tensors. Mutations to `m` then corrupt `v`. Both lists must be independently allocated with separate `zeros_like` calls.

## Faded exercise 2

The `__init__` already has `self.params = list(params)` and `self.m` allocated. Add the allocation for `self.v` (second moment buffer). It must be a SEPARATE list — do not write `self.v = self.m`. Also initialize `self.t = 0`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

class AdamState:
    def __init__(self, params):
        self.params = list(params)
        self.m = [t.zeros_like(p) for p in self.params]
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

# --- run it ---
t.manual_seed(0)
model = nn.Linear(4, 2)
adam = AdamState(model.parameters())
print(f't={adam.t}')  # 0
print(f'm[0] is v[0]: {adam.m[0] is adam.v[0]}')  # False


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(0)
    model = nn.Linear(4, 2)
    adam = AdamState(model.parameters())
    assert isinstance(adam.v, list), 'v must be a list'
    assert isinstance(adam.t, int) and adam.t == 0
    assert len(adam.v) == len(adam.m), 'v and m must have same length'
    for i, (mi, vi) in enumerate(zip(adam.m, adam.v)):
        assert id(mi) != id(vi), f'Buffer {i}: m and v must be distinct tensors (no aliasing)'
        assert mi.shape == vi.shape
    # Mutating m should NOT affect v
    adam.m[0].fill_(42.0)
    assert adam.v[0].abs().max().item() == 0.0, 'Mutating m should not change v'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class AdamState:
    def __init__(self, params):
        self.params = list(params)
        self.m = [t.zeros_like(p) for p in self.params]
        self.v = [t.zeros_like(p) for p in self.params]
        self.t = 0

# --- run it ---
t.manual_seed(0)
model = nn.Linear(4, 2)
adam = AdamState(model.parameters())
print(f't={adam.t}')  # 0
print(f'm[0] is v[0]: {adam.m[0] is adam.v[0]}')  # False
```
</details>